# 01. 데이터 로딩 · 대상지 개관

4개 지구(분당·판교·창릉·왕숙) 경계를 위성 배경에 오버레이한 인터랙티브 지도.
산출물: `output/maps/overview.html`

> 실행 커널: `AI_intro` · 필요 패키지: geopandas, folium, mapclassify

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # 프로젝트 루트 등록 → src.config import

import geopandas as gpd
import folium
from src.config import LANDUSE, ZONE_COLORS, OUT_MAPS

gdf = gpd.read_file(LANDUSE)
gdf.crs, gdf.shape

(<Geographic 2D CRS: EPSG:4326>
 Name: WGS 84
 Axis Info [ellipsoidal]:
 - Lat[north]: Geodetic latitude (degree)
 - Lon[east]: Geodetic longitude (degree)
 Area of Use:
 - name: World.
 - bounds: (-180.0, -90.0, 180.0, 90.0)
 Datum: World Geodetic System 1984 ensemble
 - Ellipsoid: WGS 84
 - Prime Meridian: Greenwich,
 (3832, 5))

In [2]:
districts = gdf.dissolve(by="zoneName").reset_index()[["zoneName", "geometry"]]
districts

,zoneName,geometry
0,고양창릉 공공주택지구,"MULTIPOLYGON Z (((126.84814 37.62176 0, 126.84..."
1,남양주왕숙 공공주택지구,"MULTIPOLYGON Z (((127.15234 37.63158 0, 127.15..."
2,성남분당,"MULTIPOLYGON Z (((127.11189 37.33557 0, 127.11..."
3,성남판교지구 택지개발사업,"MULTIPOLYGON Z (((127.07141 37.3835 0, 127.069..."


In [3]:
minx, miny, maxx, maxy = gdf.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=11, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite", overlay=False, control=True,
).add_to(m)

for _, row in districts.iterrows():
    c = ZONE_COLORS[row.zoneName]
    folium.GeoJson(
        row.geometry,
        name=row.zoneName,
        style_function=lambda x, c=c: {"color": c, "weight": 2, "fillColor": c, "fillOpacity": 0.15},
        tooltip=row.zoneName,
    ).add_to(m)

folium.LayerControl().add_to(m)
m

In [4]:
OUT_MAPS.mkdir(parents=True, exist_ok=True)
m.save(str(OUT_MAPS / "overview.html"))